# Sentiment Analysis of Customer Feedback using BERT

## Objective
In this project, we aim to classify customer feedback into positive or negative sentiment using a BERT-based deep learning model. This can help automate customer sentiment tracking and improve support quality monitoring.

We’ll use the **Customer Feedback Dataset** from Kaggle, which contains short feedback messages and their associated sentiment labels.

---

## Environment Setup

Before continuing, we verify the Python environment and install any missing libraries. We'll be using:
- Python 3.8+
- PyTorch
- Hugging Face Transformers
- pandas
- scikit-learn (for evaluation)

> Note: This notebook is intended to run inside VSCode using the Jupyter extension, with a virtual environment activated.


In [ ]:
import sys
import torch

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")

# For Colab users: install required packages
# Uncomment the line below if running in Google Colab
# !pip install transformers pandas scikit-learn torch


## 2. Load and Explore the Dataset

We will use the [Customer Feedback Dataset](https://www.kaggle.com/datasets/vishweshsalodkar/customer-feedback-dataset) from Kaggle, which contains short customer feedback texts labeled by sentiment (positive or negative).

Before training the model, we will:
- Load the dataset using `pandas`
- Check for missing or duplicated values
- Preview class distribution
- Look at a few example feedback entries


### Preprocessing Steps Applied to the CSV Data

In this notebook, the following preprocessing steps were performed on the raw customer feedback CSV:

- **Manual Parsing:** Instead of using `pd.read_csv` directly, each line was read and stripped of leading/trailing whitespace.
- **Quote Handling:** Outer double quotes were removed from lines, and the `csv.reader` was used to correctly parse fields containing commas or embedded quotes.
- **Header Extraction:** The first row was used as the column header, and subsequent rows were treated as data.
- **DataFrame Construction:** The parsed rows were assembled into a pandas DataFrame with cleaned column names.
- **Text Cleanup:** Leading and trailing double quotes in the "Text" column were removed using regular expressions.
- **Missing Data Handling:** Empty lines and rows with missing values were skipped or cleaned to ensure consistency.

These steps were necessary because the raw CSV contained formatting issues (such as extra quotes and inconsistent delimiters) that could cause errors or misinterpretation if loaded directly. Proper preprocessing ensures the data is structured and ready for analysis and modeling.


In [64]:
import pandas as pd
import csv

csv_data = "../data/raw/customer_feedback.csv"

rows = []
with open(csv_data, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:  # skip empty lines
            # Remove outer quotes if present
            if line.startswith('"') and line.endswith('"'):
                line = line[1:-1]
            # Use csv.reader to parse the inner CSV
            reader = csv.reader([line], skipinitialspace=True)
            rows.append(next(reader))

# Create DataFrame
header = rows[0]
data = rows[1:]
customer_feedback = pd.DataFrame(data, columns=[h.strip() for h in header])

# Clean up leading/trailing double quotes in the Text column
customer_feedback["Text"] = customer_feedback["Text"].str.replace('^""', '', regex=True)
customer_feedback["Text"] = customer_feedback["Text"].str.replace('""$', '', regex=True)

customer_feedback = customer_feedback.dropna(how="all")

print(f"Dataset shape: {customer_feedback.shape}")
customer_feedback.head()

Dataset shape: (96, 7)


,Text,Sentiment,Source,Date/Time,User ID,Location,Confidence Score
0,I love this product!,Positive,Twitter,2023-06-15 09:23:14,@user123,New York,0.85
1,The service was terrible.,Negative,Yelp Reviews,2023-06-15 11:45:32,user456,Los Angeles,0.65
2,This movie is amazing!,Positive,IMDb,2023-06-15 14:10:22,moviefan789,London,0.92
3,I'm so disappointed with their customer support.,Negative,Online Forum,2023-06-15 17:35:11,forumuser1,Toronto,0.78
4,Just had the best meal of my life!,Positive,TripAdvisor,2023-06-16 08:50:59,foodie22,Paris,0.88


In [67]:
# Check for missing values
customer_feedback.isnull().sum()


Text                0
Sentiment           0
Source              0
Date/Time           0
User ID             0
Location            0
Confidence Score    0
dtype: int64

## 3. Preprocessing and Label Encoding

Before we feed the data into a BERT model, we need to:
- Clean the text minimally (optional — BERT can handle raw text fairly well)
- Encode the sentiment labels into numeric form:
  - `positive` → 1
  - `negative` → 0

Since BERT uses its own tokenizer, we **do not need to lowercase, remove punctuation**, or apply stemming/lemmatization — the tokenizer handles it internally.
